In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 10_inference_pipeline_from_csv
# MAGIC 
# MAGIC **VERSIÓN QUE CARGA DIRECTAMENTE DESDE CSV ORIGINALES**
# MAGIC 
# MAGIC Esta versión:
# MAGIC - Lee los CSV de Olist directamente desde /Volumes/olist/olist_csv/olist2/
# MAGIC - Reconstruye la lógica de orders_full on-the-fly
# MAGIC - No depende de tablas Silver pre-procesadas

# COMMAND ----------

import pandas as pd
import numpy as np
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Configuración de rutas
CSV_PATH = "/Volumes/olist/olist_csv/olist2/"
MODELS_PATH = "/Volumes/olist/olist_gold/models/"
INFERENCE_PATH = "/Volumes/olist/olist_gold/inference/"

# ═══════════════════════════════════════════════════════════════
# CONFIGURACIÓN DE PERIODO DE INFERENCIA
# ═══════════════════════════════════════════════════════════════
# Estas variables se calcularán automáticamente en la Etapa 3
# pero puedes configurar el tamaño del periodo aquí:

USAR_PERIODO_AUTOMATICO = True  # True = automático, False = manual

# OPCIÓN 1: PERIODO AUTOMÁTICO (se calculará después)
# Modificar en la Etapa 3 la variable PERIODO_DIAS
# Por defecto: 90 días (3 meses)

# OPCIÓN 2: PERIODO MANUAL (si USAR_PERIODO_AUTOMATICO = False)
# Descomentar y configurar:
# START_DATE_MANUAL = "2018-06-01 00:00:00"
# END_DATE_MANUAL = "2018-08-29 23:59:59"
# CUTOFF_DATE_MANUAL = "2018-08-29 23:59:59"

START_DATE = None   # Se calculará automáticamente
END_DATE = None     # Se calculará automáticamente
CUTOFF_DATE = None  # Se calculará automáticamente

print("=" * 80)
print("🚀 PIPELINE DE INFERENCIA - DESDE CSV ORIGINALES")
print("=" * 80)
print()
print(f"📂 Fuente: {CSV_PATH}")
print(f"⚙️  Modo: {'AUTOMÁTICO (últimos 3 meses)' if USAR_PERIODO_AUTOMATICO else 'MANUAL'}")
print()

# COMMAND ----------

# MAGIC %md
# MAGIC ## ETAPA 0: Verificación de CSV y Carga de Artefactos

# COMMAND ----------

print("🔍 ETAPA 0: VERIFICACIÓN\n" + "="*80 + "\n")

# Listar archivos CSV disponibles
print("📂 Archivos CSV disponibles:")
csv_files = dbutils.fs.ls(CSV_PATH)
for file in sorted(csv_files, key=lambda x: x.name):
    size_mb = file.size / (1024 * 1024)
    print(f"   • {file.name:45s} {size_mb:>8.2f} MB")
print()

# Verificar archivos críticos
required_files = [
    "olist_orders_dataset.csv",
    "olist_order_items_dataset.csv",
    "olist_order_payments_dataset.csv",
    "olist_order_reviews_dataset.csv",
    "olist_customers_dataset.csv"
]

missing = []
for file in required_files:
    if not any(f.name == file for f in csv_files):
        missing.append(file)

if missing:
    print(f"❌ Archivos faltantes: {missing}")
    raise FileNotFoundError(f"Archivos requeridos no encontrados: {missing}")

print("✅ Todos los archivos requeridos están disponibles\n")

# Cargar artefactos del modelo
print("📦 Cargando artefactos del modelo...")
try:
    metadata = spark.read.format("delta").load(f"{MODELS_PATH}transformation_metadata/").toPandas()
    n_pca_expected = int(metadata['n_pca_components'].iloc[0])
    features_retained_df = spark.read.format("delta").load(f"{MODELS_PATH}features_retained/").toPandas()
    features_retained = features_retained_df.sort_values('order')['feature'].tolist()
    
    print(f"✅ Artefactos cargados:")
    print(f"   • Componentes PCA: {n_pca_expected}")
    print(f"   • Features retenidas: {len(features_retained)}")
    print()
except Exception as e:
    print(f"❌ Error: {e}\n⚠️  Ejecuta primero: 06b_pca_save_simple")
    raise

# COMMAND ----------

# MAGIC %md
# MAGIC ## ETAPA 1: Carga de CSV y Construcción de orders_full

# COMMAND ----------

print("📥 ETAPA 1: CARGA Y PROCESAMIENTO DE CSV\n" + "="*80 + "\n")

# 1.1 Cargar orders
print("1️⃣ Cargando olist_orders_dataset.csv...")
orders = spark.read.csv(
    f"{CSV_PATH}olist_orders_dataset.csv",
    header=True,
    inferSchema=True
)
print(f"   ✅ {orders.count():,} órdenes cargadas\n")

# 1.2 Cargar order_items
print("2️⃣ Cargando olist_order_items_dataset.csv...")
order_items = spark.read.csv(
    f"{CSV_PATH}olist_order_items_dataset.csv",
    header=True,
    inferSchema=True
)
print(f"   ✅ {order_items.count():,} items cargados\n")

# 1.3 Cargar payments
print("3️⃣ Cargando olist_order_payments_dataset.csv...")
payments = spark.read.csv(
    f"{CSV_PATH}olist_order_payments_dataset.csv",
    header=True,
    inferSchema=True
)
print(f"   ✅ {payments.count():,} pagos cargados\n")

# 1.4 Cargar reviews
print("4️⃣ Cargando olist_order_reviews_dataset.csv...")
reviews = spark.read.csv(
    f"{CSV_PATH}olist_order_reviews_dataset.csv",
    header=True,
    inferSchema=True
)
print(f"   ✅ {reviews.count():,} reviews cargadas\n")

# COMMAND ----------

# MAGIC %md
# MAGIC ## ETAPA 2: Agregar y Combinar Datos

# COMMAND ----------

print("🔧 ETAPA 2: AGREGACIÓN DE DATOS\n" + "="*80 + "\n")

# 2.1 Agregar items por orden
print("📦 Agregando items por orden...")
items_agg = order_items.groupBy("order_id").agg(
    F.count("*").alias("items_count"),
    F.countDistinct("product_id").alias("distinct_products"),
    F.sum("price").alias("sum_price"),
    F.sum("freight_value").alias("sum_freight")
)
print(f"   ✅ Agregación completada\n")

# 2.2 Agregar payments por orden
print("💳 Agregando pagos por orden...")
payments_agg = payments.groupBy("order_id").agg(
    F.sum("payment_value").alias("payment_sum"),
    F.avg("payment_installments").alias("avg_installments"),
    F.countDistinct("payment_type").alias("n_payment_types")
)
print(f"   ✅ Agregación completada\n")

# 2.3 Agregar reviews por orden
print("⭐ Agregando reviews por orden...")
reviews_agg = reviews.groupBy("order_id").agg(
    F.avg("review_score").alias("avg_review_score")
)
print(f"   ✅ Agregación completada\n")

# 2.4 Combinar todo
print("🔗 Combinando todas las fuentes...")
orders_full = orders \
    .join(items_agg, "order_id", "left") \
    .join(payments_agg, "order_id", "left") \
    .join(reviews_agg, "order_id", "left") \
    .fillna(0)

print(f"   ✅ orders_full construido: {orders_full.count():,} registros\n")

# COMMAND ----------

# MAGIC %md
# MAGIC ## ETAPA 3: Análisis y Selección de Periodo

# COMMAND ----------

print("📅 ETAPA 3: ANÁLISIS DE FECHAS DISPONIBLES\n" + "="*80 + "\n")

# Analizar rango de fechas
date_stats = orders_full.select(
    F.min('order_purchase_timestamp').alias('min_date'),
    F.max('order_purchase_timestamp').alias('max_date'),
    F.count('*').alias('total')
).collect()[0]

min_date = date_stats['min_date']
max_date = date_stats['max_date']
total_orders = date_stats['total']

print(f"📊 DATOS DISPONIBLES:")
print(f"   • Fecha mínima: {min_date}")
print(f"   • Fecha máxima: {max_date}")
print(f"   • Total órdenes: {total_orders:,}")
print()

# ═══════════════════════════════════════════════════════════════
# CONFIGURACIÓN DE PERIODO: MODIFICAR AQUÍ
# ═══════════════════════════════════════════════════════════════

from datetime import datetime, timedelta

if USAR_PERIODO_AUTOMATICO:
    # MODO AUTOMÁTICO
    if isinstance(max_date, str):
        max_dt = datetime.strptime(max_date, '%Y-%m-%d %H:%M:%S')
    else:
        max_dt = max_date
    
    # ╔═══════════════════════════════════════════════════════════╗
    # ║  CONFIGURAR TAMAÑO DEL PERIODO AQUÍ                      ║
    # ╚═══════════════════════════════════════════════════════════╝
    
    # Opciones comunes:
    # PERIODO_DIAS = 7    # 1 semana
    # PERIODO_DIAS = 14   # 2 semanas
    # PERIODO_DIAS = 30   # 1 mes
    PERIODO_DIAS = 60   # 2 meses
    # PERIODO_DIAS = 90   # 3 meses ← MODIFICAR ESTE VALOR
    # PERIODO_DIAS = 180  # 6 meses
    # PERIODO_DIAS = 365  # 1 año
    
    start_dt = max_dt - timedelta(days=PERIODO_DIAS)
    
    START_DATE = start_dt.strftime('%Y-%m-%d %H:%M:%S')
    END_DATE = max_dt.strftime('%Y-%m-%d %H:%M:%S')
    CUTOFF_DATE = END_DATE
    
    meses_aprox = PERIODO_DIAS / 30
    print(f"🤖 MODO: AUTOMÁTICO")
    print(f"🗓️  PERIODO SELECCIONADO:")
    print(f"   • Duración: {PERIODO_DIAS} días (~{meses_aprox:.1f} meses)")
    print(f"   • Inicio: {START_DATE}")
    print(f"   • Fin:    {END_DATE}")
    print(f"   • Corte:  {CUTOFF_DATE}")
    print()

else:
    # MODO MANUAL
    print(f"⚙️  MODO: MANUAL")
    
    if 'START_DATE_MANUAL' not in locals():
        raise ValueError(
            "Modo manual activado pero no se configuraron las fechas.\n"
            "Configura START_DATE_MANUAL, END_DATE_MANUAL y CUTOFF_DATE_MANUAL"
        )
    
    START_DATE = START_DATE_MANUAL
    END_DATE = END_DATE_MANUAL
    CUTOFF_DATE = CUTOFF_DATE_MANUAL
    
    print(f"🗓️  PERIODO CONFIGURADO:")
    print(f"   • Inicio: {START_DATE}")
    print(f"   • Fin:    {END_DATE}")
    print(f"   • Corte:  {CUTOFF_DATE}")
    print()

# Validar que el periodo esté dentro del rango disponible
periodo_start = datetime.strptime(START_DATE, '%Y-%m-%d %H:%M:%S')
if isinstance(min_date, str):
    data_min = datetime.strptime(min_date, '%Y-%m-%d %H:%M:%S')
else:
    data_min = min_date

if periodo_start < data_min:
    print(f"⚠️  WARNING: Periodo inicia antes de los datos disponibles")
    print(f"   Ajustando inicio a: {min_date}")
    START_DATE = min_date if isinstance(min_date, str) else min_date.strftime('%Y-%m-%d %H:%M:%S')
    print()

# COMMAND ----------

# MAGIC %md
# MAGIC ## ETAPA 4: Filtrado del Periodo

# COMMAND ----------

print("🔍 ETAPA 4: FILTRADO DEL PERIODO\n" + "="*80 + "\n")

orders_production = orders_full.filter(
    (F.col("order_purchase_timestamp") >= F.lit(START_DATE)) &
    (F.col("order_purchase_timestamp") <= F.lit(END_DATE)) &
    (F.col("order_status") != "canceled") &
    (F.col("customer_id").isNotNull())
)

n_orders = orders_production.count()
n_customers = orders_production.select("customer_id").distinct().count()

print(f"📊 RESULTADOS DEL FILTRADO:")
print(f"   • Órdenes: {n_orders:,}")
print(f"   • Clientes: {n_customers:,}")
print()

if n_orders == 0:
    print("❌ No hay órdenes en este periodo")
    print("\n💡 Ajusta manualmente el periodo o revisa los datos")
    raise ValueError("No hay órdenes en el periodo")

print(f"✅ Periodo válido con {n_orders:,} órdenes\n")

# COMMAND ----------

# MAGIC %md
# MAGIC ## ETAPA 5: Generación de Features

# COMMAND ----------

print("🎯 ETAPA 5: GENERACIÓN DE FEATURES\n" + "="*80 + "\n")

features = orders_production.groupBy("customer_id").agg(
    # RFM
    F.datediff(F.lit(CUTOFF_DATE), F.max("order_purchase_timestamp")).alias("recency"),
    F.count("order_id").alias("frequency"),
    F.sum("payment_sum").alias("monetary"),
    # Tickets
    F.avg("payment_sum").alias("avg_ticket"),
    F.max("payment_sum").alias("max_ticket"),
    F.min("payment_sum").alias("min_ticket"),
    F.stddev("payment_sum").alias("std_ticket"),
    # Items
    F.avg("items_count").alias("avg_items_per_order"),
    F.max("items_count").alias("max_items_per_order"),
    F.sum("items_count").alias("total_items"),
    F.avg("distinct_products").alias("avg_distinct_products"),
    F.sum("distinct_products").alias("total_distinct_products"),
    # Precios y flete
    F.avg("sum_price").alias("avg_price"),
    F.sum("sum_price").alias("total_price"),
    F.avg("sum_freight").alias("avg_freight"),
    F.sum("sum_freight").alias("total_freight"),
    # Pagos
    F.avg("avg_installments").alias("avg_installments"),
    F.max("avg_installments").alias("max_installments"),
    F.avg("n_payment_types").alias("avg_payment_types"),
    # Reviews
    F.avg("avg_review_score").alias("avg_review_score"),
    F.min("avg_review_score").alias("min_review_score"),
    F.max("avg_review_score").alias("max_review_score"),
    F.count(F.when(F.col("avg_review_score") != 0, 1)).alias("orders_with_review"),
    # Temporales
    F.min("order_purchase_timestamp").alias("first_purchase"),
    F.max("order_purchase_timestamp").alias("last_purchase"),
    F.datediff(F.max("order_purchase_timestamp"), F.min("order_purchase_timestamp")).alias("customer_lifetime_days"),
    # Entrega
    F.avg(F.datediff("order_delivered_customer_date", "order_purchase_timestamp")).alias("avg_delivery_days"),
    F.max(F.datediff("order_delivered_customer_date", "order_purchase_timestamp")).alias("max_delivery_days"),
    F.avg(F.datediff("order_delivered_customer_date", "order_estimated_delivery_date")).alias("avg_delay_days"),
    F.count(F.when(F.col("order_delivered_customer_date") > F.col("order_estimated_delivery_date"), 1)).alias("delayed_orders"),
    # Status
    F.count(F.when(F.col("order_status") == "delivered", 1)).alias("delivered_orders"),
    F.count(F.when(F.col("order_status") == "shipped", 1)).alias("shipped_orders")
)

# Features temporales
features = features \
    .withColumn("first_purchase_month", F.month("first_purchase")) \
    .withColumn("first_purchase_day", F.dayofmonth("first_purchase")) \
    .withColumn("first_purchase_dow", F.dayofweek("first_purchase")) \
    .withColumn("last_purchase_month", F.month("last_purchase")) \
    .withColumn("last_purchase_day", F.dayofmonth("last_purchase")) \
    .withColumn("last_purchase_dow", F.dayofweek("last_purchase")) \
    .drop("first_purchase", "last_purchase")

# Features de interacción
features = features \
    .withColumn("freight_price_ratio", 
                F.when(F.col("total_price") != 0, F.col("total_freight") / F.col("total_price")).otherwise(0)) \
    .withColumn("monetary_per_order", 
                F.when(F.col("frequency") != 0, F.col("monetary") / F.col("frequency")).otherwise(0)) \
    .withColumn("items_per_monetary", 
                F.when(F.col("monetary") != 0, F.col("total_items") / F.col("monetary")).otherwise(0)) \
    .withColumn("products_per_order", 
                F.when(F.col("frequency") != 0, F.col("total_distinct_products") / F.col("frequency")).otherwise(0)) \
    .withColumn("review_score_x_monetary", F.col("avg_review_score") * F.col("monetary")) \
    .withColumn("delayed_ratio", 
                F.when(F.col("frequency") != 0, F.col("delayed_orders") / F.col("frequency")).otherwise(0)) \
    .withColumn("delivered_ratio", 
                F.when(F.col("frequency") != 0, F.col("delivered_orders") / F.col("frequency")).otherwise(0)) \
    .withColumn("orders_per_day", 
                F.when(F.col("customer_lifetime_days") != 0, F.col("frequency") / F.col("customer_lifetime_days")).otherwise(0))

customer_features_raw = features.fillna(0).toPandas()

print(f"✅ Features generadas: {customer_features_raw.shape}\n")

# COMMAND ----------

# MAGIC %md
# MAGIC ## ETAPA 6-9: Selección, Estandarización, PCA (igual que versión anterior)

# COMMAND ----------

print("✂️  ETAPA 6: SELECCIÓN DE FEATURES\n" + "="*80 + "\n")

customer_ids = customer_features_raw['customer_id'].copy()
feature_cols_raw = [c for c in customer_features_raw.columns if c != 'customer_id']

# Alinear con features retenidas
for col in features_retained:
    if col not in customer_features_raw.columns:
        customer_features_raw[col] = 0

customer_features_selected = customer_features_raw[features_retained].copy()
print(f"✅ Features seleccionadas: {customer_features_selected.shape}\n")

# COMMAND ----------

print("📏 ETAPA 7: ESTANDARIZACIÓN\n" + "="*80 + "\n")

scaler_params_df = spark.read.format("delta").load(f"{MODELS_PATH}scaler_params/").toPandas()
scaler_params_df = scaler_params_df.sort_values('feature_index')

scaler = StandardScaler()
scaler.mean_ = scaler_params_df['mean'].values
scaler.scale_ = scaler_params_df['scale'].values
scaler.var_ = scaler_params_df['var'].values
scaler.n_features_in_ = len(scaler_params_df)

customer_features_scaled = scaler.transform(customer_features_selected)
print(f"✅ Estandarizado: {customer_features_scaled.shape}\n")

# COMMAND ----------

print("🔬 ETAPA 8: PCA\n" + "="*80 + "\n")

pca_components_df = spark.read.format("delta").load(f"{MODELS_PATH}pca_components/").toPandas()
pca_components_df = pca_components_df.sort_values('component_id')
pca_params_df = spark.read.format("delta").load(f"{MODELS_PATH}pca_params/").toPandas()
pca_params_df = pca_params_df.sort_values('component_id')
pca_mean_df = spark.read.format("delta").load(f"{MODELS_PATH}pca_mean/").toPandas()

pca = PCA(n_components=len(pca_params_df))
component_cols = [c for c in pca_components_df.columns if c != 'component_id']
pca.components_ = pca_components_df[component_cols].values
pca.explained_variance_ = pca_params_df['explained_variance'].values
pca.explained_variance_ratio_ = pca_params_df['explained_variance_ratio'].values
pca.singular_values_ = pca_params_df['singular_values'].values
pca.mean_ = pca_mean_df['pca_mean'].values
pca.n_features_in_ = len(pca.mean_)
pca.n_components_ = len(pca.components_)

X_pca = pca.transform(customer_features_scaled)

pca_cols = [f'pca_{i+1}' for i in range(pca.n_components_)]
customer_features_pca = pd.DataFrame(X_pca, columns=pca_cols)
customer_features_pca['customer_id'] = customer_ids.values

print(f"✅ PCA aplicado: {customer_features_pca.shape}\n")

# COMMAND ----------

print("✅ ETAPA 9: VALIDACIÓN Y GUARDADO\n" + "="*80 + "\n")

assert customer_features_pca.isnull().sum().sum() == 0
print("✓ Validación OK\n")

# Generar nombre dinámico basado en fechas
start_str = START_DATE[:10].replace('-', '')
end_str = END_DATE[:10].replace('-', '')
output_path = f"{INFERENCE_PATH}customer_features_pca_{start_str}_{end_str}/"

try:
    spark.sql("CREATE VOLUME IF NOT EXISTS olist.olist_gold.inference")
except:
    pass

spark.createDataFrame(customer_features_pca).write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").save(output_path)

print(f"✅ Guardado: {output_path}\n")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Resumen Final

# COMMAND ----------

print("\n" + "="*80)
print("✅ PIPELINE COMPLETADO - DESDE CSV")
print("="*80)
print(f"\n📂 Fuente: CSV originales en {CSV_PATH}")
print(f"📅 Periodo: {START_DATE} → {END_DATE}")
print(f"📊 Órdenes: {n_orders:,}")
print(f"📊 Clientes: {n_customers:,}")
print(f"📊 Features: {len(features_retained)} → {pca.n_components_} PCA")
print(f"\n💾 Output: {output_path}")
print(f"\n🎯 Dataset listo para predicción")
print("\n" + "="*80)

In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 10_inference_pipeline_from_csv
# MAGIC 
# MAGIC **VERSIÓN QUE CARGA DIRECTAMENTE DESDE CSV ORIGINALES**
# MAGIC 
# MAGIC Esta versión:
# MAGIC - Lee los CSV de Olist directamente desde /Volumes/olist/olist_csv/olist2/
# MAGIC - Reconstruye la lógica de orders_full on-the-fly
# MAGIC - No depende de tablas Silver pre-procesadas

# COMMAND ----------

import pandas as pd
import numpy as np
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Configuración de rutas
CSV_PATH = "/Volumes/olist/olist_csv/olist2/"
MODELS_PATH = "/Volumes/olist/olist_gold/models/"
INFERENCE_PATH = "/Volumes/olist/olist_gold/inference/"

# ═══════════════════════════════════════════════════════════════
# CONFIGURACIÓN DE PERIODO DE INFERENCIA
# ═══════════════════════════════════════════════════════════════
# Estas variables se calcularán automáticamente en la Etapa 3
# pero puedes configurar el tamaño del periodo aquí:

USAR_PERIODO_AUTOMATICO = True  # True = automático, False = manual

# OPCIÓN 1: PERIODO AUTOMÁTICO (se calculará después)
# Modificar en la Etapa 3 la variable PERIODO_DIAS
# Por defecto: 90 días (3 meses)

# OPCIÓN 2: PERIODO MANUAL (si USAR_PERIODO_AUTOMATICO = False)
# Descomentar y configurar:
# START_DATE_MANUAL = "2018-06-01 00:00:00"
# END_DATE_MANUAL = "2018-08-29 23:59:59"
# CUTOFF_DATE_MANUAL = "2018-08-29 23:59:59"

START_DATE = None   # Se calculará automáticamente
END_DATE = None     # Se calculará automáticamente
CUTOFF_DATE = None  # Se calculará automáticamente

print("=" * 80)
print("🚀 PIPELINE DE INFERENCIA - DESDE CSV ORIGINALES")
print("=" * 80)
print()
print(f"📂 Fuente: {CSV_PATH}")
print(f"⚙️  Modo: {'AUTOMÁTICO (últimos 3 meses)' if USAR_PERIODO_AUTOMATICO else 'MANUAL'}")
print()

# COMMAND ----------

# MAGIC %md
# MAGIC ## ETAPA 0: Verificación de CSV y Carga de Artefactos

# COMMAND ----------

print("🔍 ETAPA 0: VERIFICACIÓN\n" + "="*80 + "\n")

# Listar archivos CSV disponibles
print("📂 Archivos CSV disponibles:")
csv_files = dbutils.fs.ls(CSV_PATH)
for file in sorted(csv_files, key=lambda x: x.name):
    size_mb = file.size / (1024 * 1024)
    print(f"   • {file.name:45s} {size_mb:>8.2f} MB")
print()

# Verificar archivos críticos
required_files = [
    "olist_orders_dataset.csv",
    "olist_order_items_dataset.csv",
    "olist_order_payments_dataset.csv",
    "olist_order_reviews_dataset.csv",
    "olist_customers_dataset.csv"
]

missing = []
for file in required_files:
    if not any(f.name == file for f in csv_files):
        missing.append(file)

if missing:
    print(f"❌ Archivos faltantes: {missing}")
    raise FileNotFoundError(f"Archivos requeridos no encontrados: {missing}")

print("✅ Todos los archivos requeridos están disponibles\n")

# Cargar artefactos del modelo
print("📦 Cargando artefactos del modelo...")
try:
    metadata = spark.read.format("delta").load(f"{MODELS_PATH}transformation_metadata/").toPandas()
    n_pca_expected = int(metadata['n_pca_components'].iloc[0])
    features_retained_df = spark.read.format("delta").load(f"{MODELS_PATH}features_retained/").toPandas()
    features_retained = features_retained_df.sort_values('order')['feature'].tolist()
    
    print(f"✅ Artefactos cargados:")
    print(f"   • Componentes PCA: {n_pca_expected}")
    print(f"   • Features retenidas: {len(features_retained)}")
    print()
except Exception as e:
    print(f"❌ Error: {e}\n⚠️  Ejecuta primero: 06b_pca_save_simple")
    raise

# COMMAND ----------

# MAGIC %md
# MAGIC ## ETAPA 1: Carga de CSV y Construcción de orders_full

# COMMAND ----------

print("📥 ETAPA 1: CARGA Y PROCESAMIENTO DE CSV\n" + "="*80 + "\n")

# 1.1 Cargar orders (con schema explícito para fechas)
print("1️⃣ Cargando olist_orders_dataset.csv...")
orders = spark.read.csv(
    f"{CSV_PATH}olist_orders_dataset.csv",
    header=True,
    inferSchema=False  # ← Cambiar a False para manejar fechas manualmente
)

# Convertir columnas de fecha explícitamente
orders = orders \
    .withColumn("order_purchase_timestamp", F.to_timestamp("order_purchase_timestamp")) \
    .withColumn("order_approved_at", F.to_timestamp("order_approved_at")) \
    .withColumn("order_delivered_carrier_date", F.to_timestamp("order_delivered_carrier_date")) \
    .withColumn("order_delivered_customer_date", F.to_timestamp("order_delivered_customer_date")) \
    .withColumn("order_estimated_delivery_date", F.to_timestamp("order_estimated_delivery_date"))

print(f"   ✅ {orders.count():,} órdenes cargadas\n")

# 1.2 Cargar order_items
print("2️⃣ Cargando olist_order_items_dataset.csv...")
order_items = spark.read.csv(
    f"{CSV_PATH}olist_order_items_dataset.csv",
    header=True,
    inferSchema=True
)
print(f"   ✅ {order_items.count():,} items cargados\n")

# 1.3 Cargar payments
print("3️⃣ Cargando olist_order_payments_dataset.csv...")
payments = spark.read.csv(
    f"{CSV_PATH}olist_order_payments_dataset.csv",
    header=True,
    inferSchema=True
)
print(f"   ✅ {payments.count():,} pagos cargados\n")

# 1.4 Cargar reviews
print("4️⃣ Cargando olist_order_reviews_dataset.csv...")
reviews = spark.read.csv(
    f"{CSV_PATH}olist_order_reviews_dataset.csv",
    header=True,
    inferSchema=False  # ← Cambiar a False para fechas
)

# Convertir columnas de fecha en reviews
reviews = reviews \
    .withColumn("review_creation_date", F.to_timestamp("review_creation_date")) \
    .withColumn("review_answer_timestamp", F.to_timestamp("review_answer_timestamp")) \
    .withColumn("review_score", F.col("review_score").cast("integer"))

print(f"   ✅ {reviews.count():,} reviews cargadas\n")

# COMMAND ----------

# MAGIC %md
# MAGIC ## ETAPA 2: Agregar y Combinar Datos

# COMMAND ----------

print("🔧 ETAPA 2: AGREGACIÓN DE DATOS\n" + "="*80 + "\n")

# 2.1 Agregar items por orden
print("📦 Agregando items por orden...")
items_agg = order_items.groupBy("order_id").agg(
    F.count("*").alias("items_count"),
    F.countDistinct("product_id").alias("distinct_products"),
    F.sum("price").alias("sum_price"),
    F.sum("freight_value").alias("sum_freight")
)
print(f"   ✅ Agregación completada\n")

# 2.2 Agregar payments por orden
print("💳 Agregando pagos por orden...")
payments_agg = payments.groupBy("order_id").agg(
    F.sum("payment_value").alias("payment_sum"),
    F.avg("payment_installments").alias("avg_installments"),
    F.countDistinct("payment_type").alias("n_payment_types")
)
print(f"   ✅ Agregación completada\n")

# 2.3 Agregar reviews por orden
print("⭐ Agregando reviews por orden...")
reviews_agg = reviews.groupBy("order_id").agg(
    F.avg("review_score").alias("avg_review_score")
)
print(f"   ✅ Agregación completada\n")

# 2.4 Combinar todo
print("🔗 Combinando todas las fuentes...")
orders_full = orders \
    .join(items_agg, "order_id", "left") \
    .join(payments_agg, "order_id", "left") \
    .join(reviews_agg, "order_id", "left") \
    .fillna(0)

print(f"   ✅ orders_full construido: {orders_full.count():,} registros\n")

# COMMAND ----------

# MAGIC %md
# MAGIC ## ETAPA 3: Análisis y Selección de Periodo

# COMMAND ----------

print("📅 ETAPA 3: ANÁLISIS DE FECHAS DISPONIBLES\n" + "="*80 + "\n")

# Analizar rango de fechas
date_stats = orders_full.select(
    F.min('order_purchase_timestamp').alias('min_date'),
    F.max('order_purchase_timestamp').alias('max_date'),
    F.count('*').alias('total')
).collect()[0]

min_date = date_stats['min_date']
max_date = date_stats['max_date']
total_orders = date_stats['total']

print(f"📊 DATOS DISPONIBLES:")
print(f"   • Fecha mínima: {min_date}")
print(f"   • Fecha máxima: {max_date}")
print(f"   • Total órdenes: {total_orders:,}")
print()

# ═══════════════════════════════════════════════════════════════
# CONFIGURACIÓN DE PERIODO: MODIFICAR AQUÍ
# ═══════════════════════════════════════════════════════════════

from datetime import datetime, timedelta

if USAR_PERIODO_AUTOMATICO:
    # MODO AUTOMÁTICO
    if isinstance(max_date, str):
        max_dt = datetime.strptime(max_date, '%Y-%m-%d %H:%M:%S')
    else:
        max_dt = max_date
    
    # ╔═══════════════════════════════════════════════════════════╗
    # ║  CONFIGURAR TAMAÑO DEL PERIODO AQUÍ                      ║
    # ╚═══════════════════════════════════════════════════════════╝
    
    # Opciones comunes:
    # PERIODO_DIAS = 7    # 1 semana
    # PERIODO_DIAS = 14   # 2 semanas
    # PERIODO_DIAS = 30   # 1 mes
    PERIODO_DIAS = 60   # 2 meses ← CAMBIAR ESTE VALOR (era 90 para 3 meses)
    # PERIODO_DIAS = 90   # 3 meses
    # PERIODO_DIAS = 180  # 6 meses
    # PERIODO_DIAS = 365  # 1 año
    
    start_dt = max_dt - timedelta(days=PERIODO_DIAS)
    
    START_DATE = start_dt.strftime('%Y-%m-%d %H:%M:%S')
    END_DATE = max_dt.strftime('%Y-%m-%d %H:%M:%S')
    CUTOFF_DATE = END_DATE
    
    meses_aprox = PERIODO_DIAS / 30
    print(f"🤖 MODO: AUTOMÁTICO")
    print(f"🗓️  PERIODO SELECCIONADO:")
    print(f"   • Duración: {PERIODO_DIAS} días (~{meses_aprox:.1f} meses)")
    print(f"   • Inicio: {START_DATE}")
    print(f"   • Fin:    {END_DATE}")
    print(f"   • Corte:  {CUTOFF_DATE}")
    print()

else:
    # MODO MANUAL
    print(f"⚙️  MODO: MANUAL")
    
    if 'START_DATE_MANUAL' not in locals():
        raise ValueError(
            "Modo manual activado pero no se configuraron las fechas.\n"
            "Configura START_DATE_MANUAL, END_DATE_MANUAL y CUTOFF_DATE_MANUAL"
        )
    
    START_DATE = START_DATE_MANUAL
    END_DATE = END_DATE_MANUAL
    CUTOFF_DATE = CUTOFF_DATE_MANUAL
    
    print(f"🗓️  PERIODO CONFIGURADO:")
    print(f"   • Inicio: {START_DATE}")
    print(f"   • Fin:    {END_DATE}")
    print(f"   • Corte:  {CUTOFF_DATE}")
    print()

# Validar que el periodo esté dentro del rango disponible
periodo_start = datetime.strptime(START_DATE, '%Y-%m-%d %H:%M:%S')
if isinstance(min_date, str):
    data_min = datetime.strptime(min_date, '%Y-%m-%d %H:%M:%S')
else:
    data_min = min_date

if periodo_start < data_min:
    print(f"⚠️  WARNING: Periodo inicia antes de los datos disponibles")
    print(f"   Ajustando inicio a: {min_date}")
    START_DATE = min_date if isinstance(min_date, str) else min_date.strftime('%Y-%m-%d %H:%M:%S')
    print()

# COMMAND ----------

# MAGIC %md
# MAGIC ## ETAPA 4: Filtrado del Periodo

# COMMAND ----------

print("🔍 ETAPA 4: FILTRADO DEL PERIODO\n" + "="*80 + "\n")

orders_production = orders_full.filter(
    (F.col("order_purchase_timestamp") >= F.lit(START_DATE)) &
    (F.col("order_purchase_timestamp") <= F.lit(END_DATE)) &
    (F.col("order_status") != "canceled") &
    (F.col("customer_id").isNotNull())
)

n_orders = orders_production.count()
n_customers = orders_production.select("customer_id").distinct().count()

print(f"📊 RESULTADOS DEL FILTRADO:")
print(f"   • Órdenes: {n_orders:,}")
print(f"   • Clientes: {n_customers:,}")
print()

if n_orders == 0:
    print("❌ No hay órdenes en este periodo")
    print("\n💡 Ajusta manualmente el periodo o revisa los datos")
    raise ValueError("No hay órdenes en el periodo")

print(f"✅ Periodo válido con {n_orders:,} órdenes\n")

# COMMAND ----------

# MAGIC %md
# MAGIC ## ETAPA 5: Generación de Features

# COMMAND ----------

print("🎯 ETAPA 5: GENERACIÓN DE FEATURES\n" + "="*80 + "\n")

features = orders_production.groupBy("customer_id").agg(
    # RFM
    F.datediff(F.lit(CUTOFF_DATE), F.max("order_purchase_timestamp")).alias("recency"),
    F.count("order_id").alias("frequency"),
    F.sum("payment_sum").alias("monetary"),
    # Tickets
    F.avg("payment_sum").alias("avg_ticket"),
    F.max("payment_sum").alias("max_ticket"),
    F.min("payment_sum").alias("min_ticket"),
    F.stddev("payment_sum").alias("std_ticket"),
    # Items
    F.avg("items_count").alias("avg_items_per_order"),
    F.max("items_count").alias("max_items_per_order"),
    F.sum("items_count").alias("total_items"),
    F.avg("distinct_products").alias("avg_distinct_products"),
    F.sum("distinct_products").alias("total_distinct_products"),
    # Precios y flete
    F.avg("sum_price").alias("avg_price"),
    F.sum("sum_price").alias("total_price"),
    F.avg("sum_freight").alias("avg_freight"),
    F.sum("sum_freight").alias("total_freight"),
    # Pagos
    F.avg("avg_installments").alias("avg_installments"),
    F.max("avg_installments").alias("max_installments"),
    F.avg("n_payment_types").alias("avg_payment_types"),
    # Reviews
    F.avg("avg_review_score").alias("avg_review_score"),
    F.min("avg_review_score").alias("min_review_score"),
    F.max("avg_review_score").alias("max_review_score"),
    F.count(F.when(F.col("avg_review_score") != 0, 1)).alias("orders_with_review"),
    # Temporales
    F.min("order_purchase_timestamp").alias("first_purchase"),
    F.max("order_purchase_timestamp").alias("last_purchase"),
    F.datediff(F.max("order_purchase_timestamp"), F.min("order_purchase_timestamp")).alias("customer_lifetime_days"),
    # Entrega
    F.avg(F.datediff("order_delivered_customer_date", "order_purchase_timestamp")).alias("avg_delivery_days"),
    F.max(F.datediff("order_delivered_customer_date", "order_purchase_timestamp")).alias("max_delivery_days"),
    F.avg(F.datediff("order_delivered_customer_date", "order_estimated_delivery_date")).alias("avg_delay_days"),
    F.count(F.when(F.col("order_delivered_customer_date") > F.col("order_estimated_delivery_date"), 1)).alias("delayed_orders"),
    # Status
    F.count(F.when(F.col("order_status") == "delivered", 1)).alias("delivered_orders"),
    F.count(F.when(F.col("order_status") == "shipped", 1)).alias("shipped_orders")
)

# Features temporales
features = features \
    .withColumn("first_purchase_month", F.month("first_purchase")) \
    .withColumn("first_purchase_day", F.dayofmonth("first_purchase")) \
    .withColumn("first_purchase_dow", F.dayofweek("first_purchase")) \
    .withColumn("last_purchase_month", F.month("last_purchase")) \
    .withColumn("last_purchase_day", F.dayofmonth("last_purchase")) \
    .withColumn("last_purchase_dow", F.dayofweek("last_purchase")) \
    .drop("first_purchase", "last_purchase")

# Features de interacción
features = features \
    .withColumn("freight_price_ratio", 
                F.when(F.col("total_price") != 0, F.col("total_freight") / F.col("total_price")).otherwise(0)) \
    .withColumn("monetary_per_order", 
                F.when(F.col("frequency") != 0, F.col("monetary") / F.col("frequency")).otherwise(0)) \
    .withColumn("items_per_monetary", 
                F.when(F.col("monetary") != 0, F.col("total_items") / F.col("monetary")).otherwise(0)) \
    .withColumn("products_per_order", 
                F.when(F.col("frequency") != 0, F.col("total_distinct_products") / F.col("frequency")).otherwise(0)) \
    .withColumn("review_score_x_monetary", F.col("avg_review_score") * F.col("monetary")) \
    .withColumn("delayed_ratio", 
                F.when(F.col("frequency") != 0, F.col("delayed_orders") / F.col("frequency")).otherwise(0)) \
    .withColumn("delivered_ratio", 
                F.when(F.col("frequency") != 0, F.col("delivered_orders") / F.col("frequency")).otherwise(0)) \
    .withColumn("orders_per_day", 
                F.when(F.col("customer_lifetime_days") != 0, F.col("frequency") / F.col("customer_lifetime_days")).otherwise(0))

customer_features_raw = features.fillna(0).toPandas()

print(f"✅ Features generadas: {customer_features_raw.shape}\n")

# COMMAND ----------

# MAGIC %md
# MAGIC ## ETAPA 6-9: Selección, Estandarización, PCA (igual que versión anterior)

# COMMAND ----------

print("✂️  ETAPA 6: SELECCIÓN DE FEATURES\n" + "="*80 + "\n")

customer_ids = customer_features_raw['customer_id'].copy()
feature_cols_raw = [c for c in customer_features_raw.columns if c != 'customer_id']

# Alinear con features retenidas
for col in features_retained:
    if col not in customer_features_raw.columns:
        customer_features_raw[col] = 0

customer_features_selected = customer_features_raw[features_retained].copy()
print(f"✅ Features seleccionadas: {customer_features_selected.shape}\n")

# COMMAND ----------

print("📏 ETAPA 7: ESTANDARIZACIÓN\n" + "="*80 + "\n")

scaler_params_df = spark.read.format("delta").load(f"{MODELS_PATH}scaler_params/").toPandas()
scaler_params_df = scaler_params_df.sort_values('feature_index')

scaler = StandardScaler()
scaler.mean_ = scaler_params_df['mean'].values
scaler.scale_ = scaler_params_df['scale'].values
scaler.var_ = scaler_params_df['var'].values
scaler.n_features_in_ = len(scaler_params_df)

customer_features_scaled = scaler.transform(customer_features_selected)
print(f"✅ Estandarizado: {customer_features_scaled.shape}\n")

# COMMAND ----------

print("🔬 ETAPA 8: PCA\n" + "="*80 + "\n")

pca_components_df = spark.read.format("delta").load(f"{MODELS_PATH}pca_components/").toPandas()
pca_components_df = pca_components_df.sort_values('component_id')
pca_params_df = spark.read.format("delta").load(f"{MODELS_PATH}pca_params/").toPandas()
pca_params_df = pca_params_df.sort_values('component_id')
pca_mean_df = spark.read.format("delta").load(f"{MODELS_PATH}pca_mean/").toPandas()

pca = PCA(n_components=len(pca_params_df))
component_cols = [c for c in pca_components_df.columns if c != 'component_id']
pca.components_ = pca_components_df[component_cols].values
pca.explained_variance_ = pca_params_df['explained_variance'].values
pca.explained_variance_ratio_ = pca_params_df['explained_variance_ratio'].values
pca.singular_values_ = pca_params_df['singular_values'].values
pca.mean_ = pca_mean_df['pca_mean'].values
pca.n_features_in_ = len(pca.mean_)
pca.n_components_ = len(pca.components_)

X_pca = pca.transform(customer_features_scaled)

pca_cols = [f'pca_{i+1}' for i in range(pca.n_components_)]
customer_features_pca = pd.DataFrame(X_pca, columns=pca_cols)
customer_features_pca['customer_id'] = customer_ids.values

print(f"✅ PCA aplicado: {customer_features_pca.shape}\n")

# COMMAND ----------

print("✅ ETAPA 9: VALIDACIÓN Y GUARDADO\n" + "="*80 + "\n")

assert customer_features_pca.isnull().sum().sum() == 0
print("✓ Validación OK\n")

# Generar nombre dinámico basado en fechas
start_str = START_DATE[:10].replace('-', '')
end_str = END_DATE[:10].replace('-', '')
output_path = f"{INFERENCE_PATH}customer_features_pca_{start_str}_{end_str}/"

try:
    spark.sql("CREATE VOLUME IF NOT EXISTS olist.olist_gold.inference")
except:
    pass

spark.createDataFrame(customer_features_pca).write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").save(output_path)

print(f"✅ Guardado: {output_path}\n")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Resumen Final

# COMMAND ----------

print("\n" + "="*80)
print("✅ PIPELINE COMPLETADO - DESDE CSV")
print("="*80)
print(f"\n📂 Fuente: CSV originales en {CSV_PATH}")
print(f"📅 Periodo: {START_DATE} → {END_DATE}")
print(f"📊 Órdenes: {n_orders:,}")
print(f"📊 Clientes: {n_customers:,}")
print(f"📊 Features: {len(features_retained)} → {pca.n_components_} PCA")
print(f"\n💾 Output: {output_path}")
print(f"\n🎯 Dataset listo para predicción")
print("\n" + "="*80)

In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 10_inference_pipeline_from_csv
# MAGIC 
# MAGIC **VERSIÓN QUE CARGA DIRECTAMENTE DESDE CSV ORIGINALES**
# MAGIC 
# MAGIC Esta versión:
# MAGIC - Lee los CSV de Olist directamente desde /Volumes/olist/olist_csv/olist2/
# MAGIC - Reconstruye la lógica de orders_full on-the-fly
# MAGIC - No depende de tablas Silver pre-procesadas

# COMMAND ----------

import pandas as pd
import numpy as np
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Configuración de rutas
CSV_PATH = "/Volumes/olist/olist_csv/olist2/"
MODELS_PATH = "/Volumes/olist/olist_gold/models/"
INFERENCE_PATH = "/Volumes/olist/olist_gold/inference/"

# ═══════════════════════════════════════════════════════════════
# CONFIGURACIÓN DE PERIODO DE INFERENCIA
# ═══════════════════════════════════════════════════════════════
# Estas variables se calcularán automáticamente en la Etapa 3
# pero puedes configurar el tamaño del periodo aquí:

USAR_PERIODO_AUTOMATICO = True  # True = automático, False = manual

# OPCIÓN 1: PERIODO AUTOMÁTICO (se calculará después)
# Modificar en la Etapa 3 la variable PERIODO_DIAS
# Por defecto: 90 días (3 meses)

# OPCIÓN 2: PERIODO MANUAL (si USAR_PERIODO_AUTOMATICO = False)
# Descomentar y configurar:
# START_DATE_MANUAL = "2018-06-01 00:00:00"
# END_DATE_MANUAL = "2018-08-29 23:59:59"
# CUTOFF_DATE_MANUAL = "2018-08-29 23:59:59"

START_DATE = None   # Se calculará automáticamente
END_DATE = None     # Se calculará automáticamente
CUTOFF_DATE = None  # Se calculará automáticamente

print("=" * 80)
print("🚀 PIPELINE DE INFERENCIA - DESDE CSV ORIGINALES")
print("=" * 80)
print()
print(f"📂 Fuente: {CSV_PATH}")
print(f"⚙️  Modo: {'AUTOMÁTICO (últimos 3 meses)' if USAR_PERIODO_AUTOMATICO else 'MANUAL'}")
print()

# COMMAND ----------

# MAGIC %md
# MAGIC ## ETAPA 0: Verificación de CSV y Carga de Artefactos

# COMMAND ----------

print("🔍 ETAPA 0: VERIFICACIÓN\n" + "="*80 + "\n")

# Listar archivos CSV disponibles
print("📂 Archivos CSV disponibles:")
csv_files = dbutils.fs.ls(CSV_PATH)
for file in sorted(csv_files, key=lambda x: x.name):
    size_mb = file.size / (1024 * 1024)
    print(f"   • {file.name:45s} {size_mb:>8.2f} MB")
print()

# Verificar archivos críticos
required_files = [
    "olist_orders_dataset.csv",
    "olist_order_items_dataset.csv",
    "olist_order_payments_dataset.csv",
    "olist_order_reviews_dataset.csv",
    "olist_customers_dataset.csv"
]

missing = []
for file in required_files:
    if not any(f.name == file for f in csv_files):
        missing.append(file)

if missing:
    print(f"❌ Archivos faltantes: {missing}")
    raise FileNotFoundError(f"Archivos requeridos no encontrados: {missing}")

print("✅ Todos los archivos requeridos están disponibles\n")

# Cargar artefactos del modelo
print("📦 Cargando artefactos del modelo...")
try:
    metadata = spark.read.format("delta").load(f"{MODELS_PATH}transformation_metadata/").toPandas()
    n_pca_expected = int(metadata['n_pca_components'].iloc[0])
    features_retained_df = spark.read.format("delta").load(f"{MODELS_PATH}features_retained/").toPandas()
    features_retained = features_retained_df.sort_values('order')['feature'].tolist()
    
    print(f"✅ Artefactos cargados:")
    print(f"   • Componentes PCA: {n_pca_expected}")
    print(f"   • Features retenidas: {len(features_retained)}")
    print()
except Exception as e:
    print(f"❌ Error: {e}\n⚠️  Ejecuta primero: 06b_pca_save_simple")
    raise

# COMMAND ----------

# MAGIC %md
# MAGIC ## ETAPA 1: Carga de CSV y Construcción de orders_full

# COMMAND ----------

print("📥 ETAPA 1: CARGA Y PROCESAMIENTO DE CSV\n" + "="*80 + "\n")

# 1.1 Cargar orders (con schema explícito para fechas)
print("1️⃣ Cargando olist_orders_dataset.csv...")
orders = spark.read.csv(
    f"{CSV_PATH}olist_orders_dataset.csv",
    header=True,
    inferSchema=False  # ← Cambiar a False para manejar fechas manualmente
)

# Convertir columnas de fecha explícitamente
orders = orders \
    .withColumn("order_purchase_timestamp", F.to_timestamp("order_purchase_timestamp")) \
    .withColumn("order_approved_at", F.to_timestamp("order_approved_at")) \
    .withColumn("order_delivered_carrier_date", F.to_timestamp("order_delivered_carrier_date")) \
    .withColumn("order_delivered_customer_date", F.to_timestamp("order_delivered_customer_date")) \
    .withColumn("order_estimated_delivery_date", F.to_timestamp("order_estimated_delivery_date"))

print(f"   ✅ {orders.count():,} órdenes cargadas\n")

# 1.2 Cargar order_items
print("2️⃣ Cargando olist_order_items_dataset.csv...")
order_items = spark.read.csv(
    f"{CSV_PATH}olist_order_items_dataset.csv",
    header=True,
    inferSchema=True
)
print(f"   ✅ {order_items.count():,} items cargados\n")

# 1.3 Cargar payments
print("3️⃣ Cargando olist_order_payments_dataset.csv...")
payments = spark.read.csv(
    f"{CSV_PATH}olist_order_payments_dataset.csv",
    header=True,
    inferSchema=True
)
print(f"   ✅ {payments.count():,} pagos cargados\n")

# 1.4 Cargar reviews
print("4️⃣ Cargando olist_order_reviews_dataset.csv...")
reviews = spark.read.csv(
    f"{CSV_PATH}olist_order_reviews_dataset.csv",
    header=True,
    inferSchema=False  # ← Cambiar a False para fechas
)

# Convertir columnas de fecha en reviews
reviews = reviews \
    .withColumn("review_creation_date", F.to_timestamp("review_creation_date")) \
    .withColumn("review_answer_timestamp", F.to_timestamp("review_answer_timestamp")) \
    .withColumn("review_score", F.col("review_score").cast("integer"))

print(f"   ✅ {reviews.count():,} reviews cargadas\n")

# COMMAND ----------

# MAGIC %md
# MAGIC ## ETAPA 2: Agregar y Combinar Datos

# COMMAND ----------

print("🔧 ETAPA 2: AGREGACIÓN DE DATOS\n" + "="*80 + "\n")

# 2.1 Agregar items por orden
print("📦 Agregando items por orden...")
items_agg = order_items.groupBy("order_id").agg(
    F.count("*").alias("items_count"),
    F.countDistinct("product_id").alias("distinct_products"),
    F.sum("price").alias("sum_price"),
    F.sum("freight_value").alias("sum_freight")
)
print(f"   ✅ Agregación completada\n")

# 2.2 Agregar payments por orden
print("💳 Agregando pagos por orden...")
payments_agg = payments.groupBy("order_id").agg(
    F.sum("payment_value").alias("payment_sum"),
    F.avg("payment_installments").alias("avg_installments"),
    F.countDistinct("payment_type").alias("n_payment_types")
)
print(f"   ✅ Agregación completada\n")

# 2.3 Agregar reviews por orden
print("⭐ Agregando reviews por orden...")
reviews_agg = reviews.groupBy("order_id").agg(
    F.avg("review_score").alias("avg_review_score")
)
print(f"   ✅ Agregación completada\n")

# 2.4 Combinar todo
print("🔗 Combinando todas las fuentes...")
orders_full = orders \
    .join(items_agg, "order_id", "left") \
    .join(payments_agg, "order_id", "left") \
    .join(reviews_agg, "order_id", "left") \
    .fillna(0)

print(f"   ✅ orders_full construido: {orders_full.count():,} registros\n")

# COMMAND ----------

# MAGIC %md
# MAGIC ## ETAPA 3: Análisis y Selección de Periodo

# COMMAND ----------

print("📅 ETAPA 3: ANÁLISIS DE FECHAS DISPONIBLES\n" + "="*80 + "\n")

# Analizar rango de fechas
date_stats = orders_full.select(
    F.min('order_purchase_timestamp').alias('min_date'),
    F.max('order_purchase_timestamp').alias('max_date'),
    F.count('*').alias('total')
).collect()[0]

min_date = date_stats['min_date']
max_date = date_stats['max_date']
total_orders = date_stats['total']

print(f"📊 DATOS DISPONIBLES:")
print(f"   • Fecha mínima: {min_date}")
print(f"   • Fecha máxima: {max_date}")
print(f"   • Total órdenes: {total_orders:,}")
print()

# ═══════════════════════════════════════════════════════════════
# CONFIGURACIÓN DE PERIODO: MODIFICAR AQUÍ
# ═══════════════════════════════════════════════════════════════

from datetime import datetime, timedelta

if USAR_PERIODO_AUTOMATICO:
    # MODO AUTOMÁTICO
    if isinstance(max_date, str):
        max_dt = datetime.strptime(max_date, '%Y-%m-%d %H:%M:%S')
    else:
        max_dt = max_date
    
    # ╔═══════════════════════════════════════════════════════════╗
    # ║  CONFIGURAR TAMAÑO DEL PERIODO AQUÍ                      ║
    # ╚═══════════════════════════════════════════════════════════╝
    
    # Opciones comunes:
    # PERIODO_DIAS = 7    # 1 semana
    # PERIODO_DIAS = 14   # 2 semanas
    # PERIODO_DIAS = 30   # 1 mes
    PERIODO_DIAS = 60   # 2 meses ← CAMBIAR ESTE VALOR (era 90 para 3 meses)
    # PERIODO_DIAS = 90   # 3 meses
    # PERIODO_DIAS = 180  # 6 meses
    # PERIODO_DIAS = 365  # 1 año
    
    start_dt = max_dt - timedelta(days=PERIODO_DIAS)
    
    START_DATE = start_dt.strftime('%Y-%m-%d %H:%M:%S')
    END_DATE = max_dt.strftime('%Y-%m-%d %H:%M:%S')
    CUTOFF_DATE = END_DATE
    
    meses_aprox = PERIODO_DIAS / 30
    print(f"🤖 MODO: AUTOMÁTICO")
    print(f"🗓️  PERIODO SELECCIONADO:")
    print(f"   • Duración: {PERIODO_DIAS} días (~{meses_aprox:.1f} meses)")
    print(f"   • Inicio: {START_DATE}")
    print(f"   • Fin:    {END_DATE}")
    print(f"   • Corte:  {CUTOFF_DATE}")
    print()

else:
    # MODO MANUAL
    print(f"⚙️  MODO: MANUAL")
    
    if 'START_DATE_MANUAL' not in locals():
        raise ValueError(
            "Modo manual activado pero no se configuraron las fechas.\n"
            "Configura START_DATE_MANUAL, END_DATE_MANUAL y CUTOFF_DATE_MANUAL"
        )
    
    START_DATE = START_DATE_MANUAL
    END_DATE = END_DATE_MANUAL
    CUTOFF_DATE = CUTOFF_DATE_MANUAL
    
    print(f"🗓️  PERIODO CONFIGURADO:")
    print(f"   • Inicio: {START_DATE}")
    print(f"   • Fin:    {END_DATE}")
    print(f"   • Corte:  {CUTOFF_DATE}")
    print()

# Validar que el periodo esté dentro del rango disponible
periodo_start = datetime.strptime(START_DATE, '%Y-%m-%d %H:%M:%S')
if isinstance(min_date, str):
    data_min = datetime.strptime(min_date, '%Y-%m-%d %H:%M:%S')
else:
    data_min = min_date

if periodo_start < data_min:
    print(f"⚠️  WARNING: Periodo inicia antes de los datos disponibles")
    print(f"   Ajustando inicio a: {min_date}")
    START_DATE = min_date if isinstance(min_date, str) else min_date.strftime('%Y-%m-%d %H:%M:%S')
    print()

# COMMAND ----------

# MAGIC %md
# MAGIC ## ETAPA 4: Filtrado del Periodo

# COMMAND ----------

print("🔍 ETAPA 4: FILTRADO DEL PERIODO\n" + "="*80 + "\n")

orders_production = orders_full.filter(
    (F.col("order_purchase_timestamp") >= F.lit(START_DATE)) &
    (F.col("order_purchase_timestamp") <= F.lit(END_DATE)) &
    (F.col("order_status") != "canceled") &
    (F.col("customer_id").isNotNull())
)

n_orders = orders_production.count()
n_customers = orders_production.select("customer_id").distinct().count()

print(f"📊 RESULTADOS DEL FILTRADO:")
print(f"   • Órdenes: {n_orders:,}")
print(f"   • Clientes: {n_customers:,}")
print()

if n_orders == 0:
    print("❌ No hay órdenes en este periodo")
    print("\n💡 Ajusta manualmente el periodo o revisa los datos")
    raise ValueError("No hay órdenes en el periodo")

print(f"✅ Periodo válido con {n_orders:,} órdenes\n")

# COMMAND ----------

# MAGIC %md
# MAGIC ## ETAPA 5: Generación de Features

# COMMAND ----------

print("🎯 ETAPA 5: GENERACIÓN DE FEATURES\n" + "="*80 + "\n")

features = orders_production.groupBy("customer_id").agg(
    # RFM
    F.datediff(F.lit(CUTOFF_DATE), F.max("order_purchase_timestamp")).alias("recency"),
    F.count("order_id").alias("frequency"),
    F.sum("payment_sum").alias("monetary"),
    # Tickets
    F.avg("payment_sum").alias("avg_ticket"),
    F.max("payment_sum").alias("max_ticket"),
    F.min("payment_sum").alias("min_ticket"),
    F.stddev("payment_sum").alias("std_ticket"),
    # Items
    F.avg("items_count").alias("avg_items_per_order"),
    F.max("items_count").alias("max_items_per_order"),
    F.sum("items_count").alias("total_items"),
    F.avg("distinct_products").alias("avg_distinct_products"),
    F.sum("distinct_products").alias("total_distinct_products"),
    # Precios y flete
    F.avg("sum_price").alias("avg_price"),
    F.sum("sum_price").alias("total_price"),
    F.avg("sum_freight").alias("avg_freight"),
    F.sum("sum_freight").alias("total_freight"),
    # Pagos
    F.avg("avg_installments").alias("avg_installments"),
    F.max("avg_installments").alias("max_installments"),
    F.avg("n_payment_types").alias("avg_payment_types"),
    # Reviews
    F.avg("avg_review_score").alias("avg_review_score"),
    F.min("avg_review_score").alias("min_review_score"),
    F.max("avg_review_score").alias("max_review_score"),
    F.count(F.when(F.col("avg_review_score") != 0, 1)).alias("orders_with_review"),
    # Temporales
    F.min("order_purchase_timestamp").alias("first_purchase"),
    F.max("order_purchase_timestamp").alias("last_purchase"),
    F.datediff(F.max("order_purchase_timestamp"), F.min("order_purchase_timestamp")).alias("customer_lifetime_days"),
    # Entrega
    F.avg(F.datediff("order_delivered_customer_date", "order_purchase_timestamp")).alias("avg_delivery_days"),
    F.max(F.datediff("order_delivered_customer_date", "order_purchase_timestamp")).alias("max_delivery_days"),
    F.avg(F.datediff("order_delivered_customer_date", "order_estimated_delivery_date")).alias("avg_delay_days"),
    F.count(F.when(F.col("order_delivered_customer_date") > F.col("order_estimated_delivery_date"), 1)).alias("delayed_orders"),
    # Status
    F.count(F.when(F.col("order_status") == "delivered", 1)).alias("delivered_orders"),
    F.count(F.when(F.col("order_status") == "shipped", 1)).alias("shipped_orders")
)

# Features temporales
features = features \
    .withColumn("first_purchase_month", F.month("first_purchase")) \
    .withColumn("first_purchase_day", F.dayofmonth("first_purchase")) \
    .withColumn("first_purchase_dow", F.dayofweek("first_purchase")) \
    .withColumn("last_purchase_month", F.month("last_purchase")) \
    .withColumn("last_purchase_day", F.dayofmonth("last_purchase")) \
    .withColumn("last_purchase_dow", F.dayofweek("last_purchase")) \
    .drop("first_purchase", "last_purchase")

# Features de interacción
features = features \
    .withColumn("freight_price_ratio", 
                F.when(F.col("total_price") != 0, F.col("total_freight") / F.col("total_price")).otherwise(0)) \
    .withColumn("monetary_per_order", 
                F.when(F.col("frequency") != 0, F.col("monetary") / F.col("frequency")).otherwise(0)) \
    .withColumn("items_per_monetary", 
                F.when(F.col("monetary") != 0, F.col("total_items") / F.col("monetary")).otherwise(0)) \
    .withColumn("products_per_order", 
                F.when(F.col("frequency") != 0, F.col("total_distinct_products") / F.col("frequency")).otherwise(0)) \
    .withColumn("review_score_x_monetary", F.col("avg_review_score") * F.col("monetary")) \
    .withColumn("delayed_ratio", 
                F.when(F.col("frequency") != 0, F.col("delayed_orders") / F.col("frequency")).otherwise(0)) \
    .withColumn("delivered_ratio", 
                F.when(F.col("frequency") != 0, F.col("delivered_orders") / F.col("frequency")).otherwise(0)) \
    .withColumn("orders_per_day", 
                F.when(F.col("customer_lifetime_days") != 0, F.col("frequency") / F.col("customer_lifetime_days")).otherwise(0))

customer_features_raw = features.fillna(0).toPandas()

print(f"✅ Features generadas: {customer_features_raw.shape}\n")

# COMMAND ----------

# MAGIC %md
# MAGIC ## ETAPA 6-9: Selección, Estandarización, PCA (igual que versión anterior)

# COMMAND ----------

print("✂️  ETAPA 6: SELECCIÓN DE FEATURES\n" + "="*80 + "\n")

customer_ids = customer_features_raw['customer_id'].copy()
feature_cols_raw = [c for c in customer_features_raw.columns if c != 'customer_id']

# Alinear con features retenidas
for col in features_retained:
    if col not in customer_features_raw.columns:
        customer_features_raw[col] = 0

customer_features_selected = customer_features_raw[features_retained].copy()
print(f"✅ Features seleccionadas: {customer_features_selected.shape}\n")

# COMMAND ----------

print("📏 ETAPA 7: ESTANDARIZACIÓN\n" + "="*80 + "\n")

scaler_params_df = spark.read.format("delta").load(f"{MODELS_PATH}scaler_params/").toPandas()
scaler_params_df = scaler_params_df.sort_values('feature_index')

scaler = StandardScaler()
scaler.mean_ = scaler_params_df['mean'].values
scaler.scale_ = scaler_params_df['scale'].values
scaler.var_ = scaler_params_df['var'].values
scaler.n_features_in_ = len(scaler_params_df)

customer_features_scaled = scaler.transform(customer_features_selected)
print(f"✅ Estandarizado: {customer_features_scaled.shape}\n")

# COMMAND ----------

print("🔬 ETAPA 8: PCA\n" + "="*80 + "\n")

pca_components_df = spark.read.format("delta").load(f"{MODELS_PATH}pca_components/").toPandas()
pca_components_df = pca_components_df.sort_values('component_id')
pca_params_df = spark.read.format("delta").load(f"{MODELS_PATH}pca_params/").toPandas()
pca_params_df = pca_params_df.sort_values('component_id')
pca_mean_df = spark.read.format("delta").load(f"{MODELS_PATH}pca_mean/").toPandas()

pca = PCA(n_components=len(pca_params_df))
component_cols = [c for c in pca_components_df.columns if c != 'component_id']
pca.components_ = pca_components_df[component_cols].values
pca.explained_variance_ = pca_params_df['explained_variance'].values
pca.explained_variance_ratio_ = pca_params_df['explained_variance_ratio'].values
pca.singular_values_ = pca_params_df['singular_values'].values
pca.mean_ = pca_mean_df['pca_mean'].values
pca.n_features_in_ = len(pca.mean_)
pca.n_components_ = len(pca.components_)

X_pca = pca.transform(customer_features_scaled)

pca_cols = [f'pca_{i+1}' for i in range(pca.n_components_)]
customer_features_pca = pd.DataFrame(X_pca, columns=pca_cols)
customer_features_pca['customer_id'] = customer_ids.values

print(f"✅ PCA aplicado: {customer_features_pca.shape}\n")

# COMMAND ----------

print("✅ ETAPA 9: VALIDACIÓN Y GUARDADO\n" + "="*80 + "\n")

assert customer_features_pca.isnull().sum().sum() == 0
print("✓ Validación OK\n")

# Generar nombre dinámico basado en fechas
start_str = START_DATE[:10].replace('-', '')
end_str = END_DATE[:10].replace('-', '')
output_path = f"{INFERENCE_PATH}customer_features_pca_{start_str}_{end_str}/"

try:
    spark.sql("CREATE VOLUME IF NOT EXISTS olist.olist_gold.inference")
except:
    pass

spark.createDataFrame(customer_features_pca).write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").save(output_path)

print(f"✅ Guardado: {output_path}\n")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Resumen Final

# COMMAND ----------

print("\n" + "="*80)
print("✅ PIPELINE COMPLETADO - DESDE CSV")
print("="*80)
print(f"\n📂 Fuente: CSV originales en {CSV_PATH}")
print(f"📅 Periodo: {START_DATE} → {END_DATE}")
print(f"📊 Órdenes: {n_orders:,}")
print(f"📊 Clientes: {n_customers:,}")
print(f"📊 Features: {len(features_retained)} → {pca.n_components_} PCA")
print(f"\n💾 Output: {output_path}")
print(f"\n🎯 Dataset listo para predicción")
print("\n" + "="*80)